In [1]:
import numpy as np
import pandas as pd
from scipy.linalg import cholesky, inv
from typing import Tuple, Optional, List

# ───────────────────────────────────────────────────────────────
# 1. Generate the random matrix
# ───────────────────────────────────────────────────────────────
np.random.seed(42)  # optional: set for reproducibility; remove for fresh randomness
raw = np.random.normal(loc=0, scale=5, size=(10, 10))

# Round to one decimal place (四捨五入到小數點第一位)
A = np.round(raw, 1)

# Display the matrix
df = pd.DataFrame(A)

In [2]:
latex_code = df.to_latex(index=False)  # index=False 可移除行號
print(latex_code)

\begin{tabular}{rrrrrrrrrr}
\toprule
   0 &    1 &    2 &    3 &     4 &    5 &    6 &    7 &    8 &    9 \\
\midrule
 2.5 & -0.7 &  3.2 &  7.6 &  -1.2 & -1.2 &  7.9 &  3.8 & -2.3 &  2.7 \\
-2.3 & -2.3 &  1.2 & -9.6 &  -8.6 & -2.8 & -5.1 &  1.6 & -4.5 & -7.1 \\
 7.3 & -1.1 &  0.3 & -7.1 &  -2.7 &  0.6 & -5.8 &  1.9 & -3.0 & -1.5 \\
-3.0 &  9.3 & -0.1 & -5.3 &   4.1 & -6.1 &  1.0 & -9.8 & -6.6 &  1.0 \\
 3.7 &  0.9 & -0.6 & -1.5 &  -7.4 & -3.6 & -2.3 &  5.3 &  1.7 & -8.8 \\
 1.6 & -1.9 & -3.4 &  3.1 &   5.2 &  4.7 & -4.2 & -1.5 &  1.7 &  4.9 \\
-2.4 & -0.9 & -5.5 & -6.0 &   4.1 &  6.8 & -0.4 &  5.0 &  1.8 & -3.2 \\
 1.8 &  7.7 & -0.2 &  7.8 & -13.1 &  4.1 &  0.4 & -1.5 &  0.5 & -9.9 \\
-1.1 &  1.8 &  7.4 & -2.6 &  -4.0 & -2.5 &  4.6 &  1.6 & -2.6 &  2.6 \\
 0.5 &  4.8 & -3.5 & -1.6 &  -2.0 & -7.3 &  1.5 &  1.3 &  0.0 & -1.2 \\
\bottomrule
\end{tabular}



In [3]:
# ───────────────────────────────────────────────────────────────
# 2. Compute the SVD of matrix `a`
# ───────────────────────────────────────────────────────────────

U, Sigma, VT = np.linalg.svd(A, full_matrices=False)
Sigma
#print("Singular values σ (length =", len(s), "):", s)

array([25.91247308, 21.24529444, 20.26523938, 15.14766336, 10.75280588,
       10.01757449,  7.53178401,  4.37201631,  3.40367867,  1.1435088 ])

---

In [4]:
def column_normalize(A: np.ndarray, eps: float = 1e-12) -> Tuple[np.ndarray, np.ndarray]:
    """
    對任意 m×n 實矩陣 A 進行欄向量 ℓ₂-norm 正規化。
    回傳 (A_hat, D)：
      A_hat ─ 每個 column 向量均除以其 ℓ₂-norm（若 norm < eps 則保留原值）。
      D     ─ n×n 對角矩陣，對角線即原始 column norms。
    """
    col_norms = np.linalg.norm(A, axis=0)         # shape (n,)
    D         = np.diag(col_norms)                # n×n diagonal matrix
    safe_norms = np.where(col_norms < eps, 1.0, col_norms)
    A_hat     = A / safe_norms                    # broadcasting
    A_hat[:, col_norms < eps] = A[:, col_norms < eps]  # 保留零向量
    return A_hat, D

def diag_inv(D: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    """Return D^{-1} for a *diagonal* 2‑D matrix `D`.
    If any diagonal entry |d_i| < eps, the corresponding inverse is set to `inf`.
    The function is self‑contained; no external helper required.
    """
    if D.ndim != 2 or D.shape[0] != D.shape[1]:
        raise ValueError("Input must be a square 2‑D matrix.")
    if not np.allclose(D, np.diag(np.diag(D))):
        raise ValueError("Input matrix is not diagonal.")
    d = np.diag(D)
    d_safe = np.where(np.abs(d) < eps, np.inf, d)
    return np.diag(1.0 / d_safe)

def col_norm_diag(A: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    """Return a *diagonal matrix* whose entries are the ℓ₂‑norms of each column of `A`.
    Columns with norm < eps get a zero on the diagonal.
    """
    col_norms = np.linalg.norm(A, axis=0)
    col_norms[col_norms < eps] = 0.0
    return np.diag(col_norms)

def random_10x10_fro_norm(target_norm: float = 1.0, seed: Optional[int] = None) -> np.ndarray:
    """Generate a 10×10 real matrix with i.i.d. 𝒩(0,1) entries and
    *exact* Frobenius norm equal to `target_norm`.

    Parameters
    ----------
    target_norm : float
        Desired Frobenius norm (> 0).
    seed : int | None
        RNG seed for reproducibility.
    """
    if target_norm <= 0:
        raise ValueError("target_norm must be positive.")
    rng = np.random.default_rng(seed)
    A = rng.normal(loc=0.0, scale=1.0, size=(10, 10))  # μ=0, σ=1
    norm = np.linalg.norm(A, ord='fro')
    if norm == 0:
        # Extremely unlikely; regenerate
        return random_10x10_fro_norm(target_norm, seed=seed)
    A *= target_norm / norm  # scale so that ‖A‖_F == target_norm
    return A

def thesis(Delta_A):
    A_tilde = A + Delta_A
    W = A_tilde @ VT.T
    W_hat, w = column_normalize(W) # w is a diagonal matrix
    WW = W_hat.T @ W_hat
    L = cholesky(WW, lower=True)  # M = L Lᵀ
    N_hat = inv(L).T        # = L^{−T}
    S_hat = diag_inv(w) @ N_hat
    Lambda = col_norm_diag(S_hat)
    U_prime, Sigma_prime, V_prime_T = np.linalg.svd(S_hat @ Lambda, full_matrices=False)
    
    new_V = VT.T @ (U_prime @ V_prime_T)
    new_U = W_hat @ N_hat
    residual = (A_tilde @ new_V) @ Lambda - new_U
    return np.linalg.norm(residual, ord='fro')

In [5]:
POWERS_100_TO_1E_13 = [10**p for p in range(2, -13, -1)]  # 100,10,1,0.1,…,1e‑12


def first_order_table(thesis_fn, magnitudes: Optional[List[float]] = None,
                      n_trials: int = 10000, seed: int = 0) -> pd.DataFrame:
    """Run `thesis_fn` on random perturbations of prescribed magnitudes.

    Parameters
    ----------
    thesis_fn : callable
        Function that maps ΔA → residual (scalar).
    magnitudes : List[float] | None
        List of Frobenius norms for ΔA. Default = [100, 10, 1, 0.1, …, 1e‑13].
    n_trials : int
        Number of random samples for each magnitude (default 1000).
    seed : int
        Base RNG seed for reproducibility.

    Returns
    -------
    DataFrame with columns ["ΔA_norm", "mean_residual", "ratio = mean_res/ΔA_norm"].
    """
    if magnitudes is None:
        magnitudes = POWERS_100_TO_1E_13

    rng = np.random.default_rng(seed)
    rows = []
    for t in magnitudes:
        res_accum = 0.0
        for _ in range(n_trials):
            ΔA = random_10x10_fro_norm(t, seed=rng.integers(1 << 31))
            res_accum += thesis_fn(ΔA)
        mean_res = res_accum / n_trials
        rows.append({"ΔA_norm": t, "mean_residual": mean_res})

    return pd.DataFrame(rows)

In [6]:
df = first_order_table(thesis)
display(df.set_index("ΔA_norm"))

,mean_residual
ΔA_norm,
1.000000e+02,3.877525e+00
1.000000e+01,4.269070e-01
1.000000e+00,3.569251e-02
1.000000e-01,3.572547e-03
1.000000e-02,3.563517e-04
1.000000e-03,3.578657e-05
1.000000e-04,3.570897e-06
1.000000e-05,3.571758e-07
1.000000e-06,3.566135e-08
